# Generator Metrics — DeepEval + RAGAS Side-by-Side

This notebook covers **generator evaluation metrics** from both **DeepEval** and **RAGAS**, measuring whether the LLM's generated answers are faithful, relevant, and correct.

### What We Cover

| Framework | Metric | What It Measures |
|-----------|--------|------------------|
| DeepEval | AnswerRelevancyMetric | Is the answer relevant to the question? |
| DeepEval | FaithfulnessMetric | Is every claim supported by the context? |
| DeepEval | HallucinationMetric | Does the answer contradict the context? |
| RAGAS | ResponseRelevancy | Is the answer relevant? (reverse-question approach) |
| RAGAS | Faithfulness | Is every claim supported by the context? |
| RAGAS | FactualCorrectness | Is the answer factually correct vs reference? |
| RAGAS | SemanticSimilarity | How semantically close is the answer to the reference? |

-
## 1. Setup & Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(dotenv_path)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# DeepEval imports
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
from deepeval.test_case import LLMTestCase
from deepeval import assert_test

# RAGAS imports
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    FactualCorrectness,
    SemanticSimilarity,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("All imports successful.")

-
## 2. Load Pipeline Results

In [ ]:
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")

if os.path.exists(data_path):
    with open(data_path) as f:
        pipeline_results = json.load(f)
    print(f"Loaded {len(pipeline_results)} results from {data_path}")
else:
    print("pipeline_results.json not found. Using inline test data.")
    pipeline_results = None

In [ ]:
# Fallback inline data
if pipeline_results is None:
    pipeline_results = [
        {
            "query": "What is the return policy for regular items?",
            "expected_output": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds processed in 5-7 business days.",
            "actual_output": "Acme Corp has a 30-day return policy for all products. Items need to be unused and in original packaging with proof of purchase. Refunds are processed within 5-7 business days. Shipping costs are the customer's responsibility unless the item was defective.",
            "retrieval_context": [
                "Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. To be eligible for a return, the item must be unused, in its original packaging, and accompanied by the original receipt or proof of purchase. Refunds are processed within 5-7 business days after we receive the returned item. Shipping costs for returns are the responsibility of the customer unless the item was defective or the wrong item was shipped.",
                "Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days.",
                "Orders can be cancelled within 1 hour of placement by contacting customer support."
            ]
        },
        {
            "query": "How long do I have to return electronics?",
            "expected_output": "Electronics have a 15-day return window with all original accessories. 15% restocking fee may apply.",
            "actual_output": "Electronic products have a 15-day return window. You must include all original accessories, cables, manuals, and packaging. A 15% restocking fee may apply to opened electronics. Defective electronics can be exchanged within 90 days.",
            "retrieval_context": [
                "Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days. All electronics must be returned with all original accessories, cables, manuals, and packaging. A restocking fee of 15% may apply to opened electronics. Defective electronics can be exchanged for the same model within the first 90 days.",
                "Acme Corp offers a 30-day return policy on all products.",
                "The Acme SmartHome Hub is our flagship home automation controller."
            ]
        },
        {
            "query": "What shipping options are available?",
            "expected_output": "Standard (5-7 days, free over $50), Expedited (2-3 days, $12.99), Overnight (next day, $24.99).",
            "actual_output": "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping (2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 before 2 PM EST).",
            "retrieval_context": [
                "Acme Corp offers several shipping options. Standard Shipping takes 5-7 business days and is free for orders over $50. Expedited Shipping takes 2-3 business days and costs $12.99. Overnight Shipping is available for $24.99.",
                "Acme Corp ships internationally to over 50 countries.",
                "The Acme Rewards program is free to join."
            ]
        },
        {
            "query": "Do you ship internationally?",
            "expected_output": "Yes, ships to 50+ countries. Standard 10-21 days, Express 5-7 days. Customer pays customs.",
            "actual_output": "Yes, Acme Corp ships internationally to over 50 countries. Standard takes 10-21 business days, Express takes 5-7 days. Customs duties and import fees are the customer's responsibility.",
            "retrieval_context": [
                "Acme Corp ships internationally to over 50 countries. International Standard Shipping takes 10-21 business days. International Express Shipping takes 5-7 business days. All international orders may be subject to customs duties.",
                "Standard Shipping takes 5-7 business days and is free for orders over $50.",
                "Acme Corp collects personal information including name and email."
            ]
        },
        {
            "query": "What does the warranty cover?",
            "expected_output": "1-year limited warranty for defects in materials and workmanship. Does not cover accidents. Extended plans available.",
            "actual_output": "All Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship under normal use. It does not cover accidents, misuse, or unauthorized modifications. Extended warranty plans of 2 and 3 years are available.",
            "retrieval_context": [
                "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship under normal use. The warranty does not cover damage caused by accidents, misuse, unauthorized modifications, or normal wear and tear. Extended warranty plans (2-year and 3-year) are available.",
                "The ErgoDesk Pro comes with a 5-year warranty on frame and motors.",
                "Customer support is available Monday through Friday."
            ]
        },
        {
            "query": "How can I contact customer support?",
            "expected_output": "Phone: 1-800-ACME-HELP (M-F 8-8 EST). Email: support@acmecorp.com. Live chat: M-Sat 9-6 EST.",
            "actual_output": "You can contact Acme Corp support by phone at 1-800-ACME-HELP (Mon-Fri 8AM-8PM EST), email at support@acmecorp.com (24-48hr response), or live chat on the website (Mon-Sat 9AM-6PM EST). There's also an automated help center at help.acmecorp.com.",
            "retrieval_context": [
                "Acme Corp customer support is available through multiple channels. Phone support is available Monday through Friday, 8 AM to 8 PM EST at 1-800-ACME-HELP. Email support at support@acmecorp.com with 24-48 hour response. Live chat Monday through Saturday, 9 AM to 6 PM EST.",
                "Orders can be cancelled within 1 hour of placement.",
                "Acme Corp collects personal information."
            ]
        },
        {
            "query": "What are the SmartHome Hub features?",
            "expected_output": "$149.99, WiFi/Bluetooth/Zigbee/Z-Wave, voice control, 5-inch touchscreen, energy monitoring, 10,000+ devices, 2-year warranty.",
            "actual_output": "The SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, and Z-Wave. Features include voice control, 5-inch touchscreen, energy monitoring, and automated routines. Compatible with 10,000+ devices. Comes with a 2-year warranty.",
            "retrieval_context": [
                "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99. It supports WiFi, Bluetooth, Zigbee, and Z-Wave protocols, compatible with over 10,000 smart home devices. Features include voice control, 5-inch touchscreen display, energy monitoring dashboard, and automated routines. 2-year warranty.",
                "The AirPure Pro is a premium air purifier priced at $299.99.",
                "Acme Corp offers a 30-day return policy."
            ]
        },
        {
            "query": "How much is the AirPure Pro and what does it filter?",
            "expected_output": "$299.99. 4-stage filtration: pre-filter, carbon, HEPA H13, UV-C. Removes 99.97% of 0.3 micron particles.",
            "actual_output": "The AirPure Pro costs $299.99 and features a 4-stage filtration system: pre-filter, activated carbon, True HEPA H13, and UV-C light sanitizer. It removes 99.97% of particles as small as 0.3 microns including dust, pollen, pet dander, mold, and smoke.",
            "retrieval_context": [
                "The Acme AirPure Pro is a premium air purifier designed for rooms up to 800 square feet, priced at $299.99. It features a 4-stage filtration system: pre-filter, activated carbon filter, True HEPA H13 filter, and UV-C light sanitizer. Removes 99.97% of particles as small as 0.3 microns.",
                "The SmartHome Hub is priced at $149.99.",
                "The FitBand Ultra is a fitness tracker priced at $79.99."
            ]
        },
        {
            "query": "What payment methods do you accept?",
            "expected_output": "Visa, MasterCard, AmEx, Discover, PayPal, Apple Pay, Google Pay, Gift Cards. Pay Later for orders $200+.",
            "actual_output": "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. Acme Pay Later is available for orders over $200, splitting into 4 interest-free installments.",
            "retrieval_context": [
                "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. For orders over $200, Acme Pay Later splits payments into 4 interest-free installments over 6 weeks.",
                "The Acme Rewards program is free to join.",
                "Acme Corp offers a 30-day return policy."
            ]
        },
        {
            "query": "How does the loyalty program work?",
            "expected_output": "Free, 1 point/dollar. 100 pts = $5. Silver (500+): free expedited shipping. Gold (1000+): priority support.",
            "actual_output": "Acme Rewards is free to join. Earn 1 point per dollar, redeem 100 points for $5 off. Silver tier (500+ points/year) gets free expedited shipping. Gold tier (1000+ points/year) gets priority support and exclusive previews. Points expire after 12 months of inactivity.",
            "retrieval_context": [
                "The Acme Rewards program is free to join and earns 1 point per dollar. 100 points = $5 off. Silver tier (500+) unlocks free expedited shipping. Gold tier (1000+) adds priority support and exclusive previews. Points expire after 12 months of inactivity.",
                "Acme Corp accepts Visa, MasterCard, and other payment methods.",
                "Standard Shipping takes 5-7 business days."
            ]
        },
        {
            "query": "Can I cancel my order?",
            "expected_output": "Within 1 hour via support or dashboard. After 1 hour may not be possible. Shipped orders must be returned.",
            "actual_output": "You can cancel within 1 hour of placing your order through customer support or the account dashboard. After 1 hour, the order may not be cancellable. Shipped orders must be returned instead. Custom items cannot be cancelled once production begins.",
            "retrieval_context": [
                "Orders can be cancelled within 1 hour of placement by contacting customer support or using the Cancel Order button in your account dashboard. After 1 hour, orders enter processing and may not be cancellable. Shipped orders cannot be cancelled and must be returned.",
                "Acme Corp offers a 30-day return policy.",
                "Customer support is available Monday through Friday."
            ]
        },
        {
            "query": "Tell me about the ErgoDesk Pro.",
            "expected_output": "$599.99, 60x30 bamboo, dual-motor, 25.5-51 inch range, 4 presets, 300 lb capacity, 5-year frame warranty.",
            "actual_output": "The ErgoDesk Pro ($599.99) has a 60x30 bamboo desktop, dual-motor height adjustment from 25.5 to 51 inches, 4 programmable presets, cable management, and anti-collision technology. Supports 300 lbs. 5-year warranty on frame/motors, 2-year on electronics.",
            "retrieval_context": [
                "The Acme ErgoDesk Pro is a motorized standing desk priced at $599.99. 60x30 inch bamboo desktop, dual-motor height adjustment from 25.5 to 51 inches, 4 programmable presets, cable management tray, anti-collision technology. Supports up to 300 lbs. 5-year warranty on frame and motors, 2-year on electronics.",
                "All Acme Corp products come with a 1-year limited warranty.",
                "The SmartHome Hub is priced at $149.99."
            ]
        }
    ]
    print(f"Using {len(pipeline_results)} inline test cases.")

In [ ]:
# Create DeepEval test cases
test_cases = []
for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        retrieval_context=r["retrieval_context"],
        expected_output=r["expected_output"],
    )
    test_cases.append(tc)

print(f"Created {len(test_cases)} test cases.")

---
### RAGAS Setup

Create RAGAS `SingleTurnSample` objects and configure the evaluator.

In [ ]:
# Configure RAGAS evaluator LLM and embeddings
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# Create RAGAS SingleTurnSample objects
ragas_samples = []
for r in pipeline_results:
    sample = SingleTurnSample(
        user_input=r["query"],
        response=r["actual_output"],
        retrieved_contexts=r["retrieval_context"],
        reference=r.get("expected_output", ""),
    )
    ragas_samples.append(sample)

ragas_dataset = EvaluationDataset(samples=ragas_samples)
print(f"Created {len(ragas_samples)} RAGAS samples.")

---
## DeepEval Generator Metrics

-
## 3. AnswerRelevancyMetric

### What It Measures

**Answer Relevancy** evaluates whether the generated answer actually addresses the user's question. An answer can be factually correct but still irrelevant if it does not answer what was asked.

### Algorithm Walkthrough

1. The LLM examines the input query and the actual output.
2. It extracts the key statements/claims from the actual output.
3. For each statement, it assesses whether the statement is relevant to the input query.
4. The final score is the proportion of statements that are relevant.

**Score = (relevant statements) / (total statements in actual output)**

### When to Use
Always include this metric. It catches cases where the LLM goes off-topic, provides unnecessary information, or misunderstands the question.

In [ ]:
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {answer_relevancy.__class__.__name__}")
print(f"Threshold: {answer_relevancy.threshold}")

In [ ]:
# Run on all test cases
ar_scores = []
ar_reasons = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    answer_relevancy.measure(tc)
    ar_scores.append(answer_relevancy.score)
    ar_reasons.append(answer_relevancy.reason)
    print(f"  Score: {answer_relevancy.score:.4f}")

print(f"\nAverage Answer Relevancy: {np.mean(ar_scores):.4f}")

In [ ]:
ar_df = pd.DataFrame({
    "Query": [tc.input[:50] + "..." for tc in test_cases],
    "Score": ar_scores,
    "Passed": [s >= 0.7 for s in ar_scores],
    "Reason": [r[:120] + "..." if len(r) > 120 else r for r in ar_reasons],
})

print(ar_df.to_string(index=False))

### Demonstrating Effect of Different Prompt Templates

The system prompt used in the RAG generator can significantly affect answer relevancy. Let's compare a focused prompt vs a verbose prompt.

In [ ]:
# Focused answer — directly addresses the question
focused_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy. Items must be unused and in original packaging with proof of purchase. Refunds are processed in 5-7 business days.",
)

# Verbose answer — includes lots of tangential information
verbose_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Thank you for your interest in our return policy! Acme Corp was founded in 1985 and has been serving customers for decades. We pride ourselves on excellent customer service. Our return policy is 30 days. You can also check out our new SmartHome Hub which is on sale this week. We also offer international shipping to 50 countries. Anyway, items must be unused for a return.",
)

ar_metric = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini")

ar_metric.measure(focused_case)
focused_score = ar_metric.score

ar_metric.measure(verbose_case)
verbose_score = ar_metric.score

print("Prompt Template Effect on Answer Relevancy:")
print(f"  Focused answer:  {focused_score:.4f}")
print(f"  Verbose answer:  {verbose_score:.4f}")
print(f"\nA focused prompt produces more relevant answers with less noise.")

-
## 4. FaithfulnessMetric

### What It Measures

**Faithfulness** evaluates whether every claim in the generated answer can be traced back to the retrieved contexts. It is the core metric for detecting hallucination in RAG systems.

### Algorithm Walkthrough

1. The actual output is decomposed into individual claims/statements ("truths").
2. Each claim is checked against the retrieval contexts.
3. A claim is "faithful" if it can be directly inferred from the contexts.
4. The score is the proportion of faithful claims.

**Score = (claims supported by contexts) / (total claims in actual output)**

### When to Use
Always include this metric for RAG evaluation. It is the primary defense against hallucination. A high faithfulness score means the LLM is sticking to the provided evidence.

In [ ]:
faithfulness = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {faithfulness.__class__.__name__}")
print(f"Threshold: {faithfulness.threshold}")

In [ ]:
# Run on all test cases
faith_scores = []
faith_reasons = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    faithfulness.measure(tc)
    faith_scores.append(faithfulness.score)
    faith_reasons.append(faithfulness.reason)
    print(f"  Score: {faithfulness.score:.4f}")

print(f"\nAverage Faithfulness: {np.mean(faith_scores):.4f}")

In [ ]:
faith_df = pd.DataFrame({
    "Query": [tc.input[:50] + "..." for tc in test_cases],
    "Score": faith_scores,
    "Passed": [s >= 0.7 for s in faith_scores],
    "Reason": [r[:120] + "..." if len(r) > 120 else r for r in faith_reasons],
})

print(faith_df.to_string(index=False))

### Demonstrating Faithfulness: Catching Hallucination

Let's create a test case where the LLM hallucinates information not in the context, and see if Faithfulness catches it.

In [ ]:
# Faithful answer — everything is from the context
faithful_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy. Items must be unused and in original packaging.",
    retrieval_context=[
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with proof of purchase."
    ],
)

# Hallucinated answer — contains fabricated claims
hallucinated_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy. Items must be unused and in original packaging. Additionally, Acme Corp offers a special 90-day holiday return window from November through January, and VIP members get a full year to return any item.",
    retrieval_context=[
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with proof of purchase."
    ],
)

faith_metric = FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini")

faith_metric.measure(faithful_case)
faithful_score = faith_metric.score
faithful_reason = faith_metric.reason

faith_metric.measure(hallucinated_case)
hallucinated_score = faith_metric.score
hallucinated_reason = faith_metric.reason

print("Faithfulness: Catching Hallucination")
print("=" * 50)
print(f"Faithful answer score:     {faithful_score:.4f}")
print(f"  Reason: {faithful_reason[:150]}")
print(f"\nHallucinated answer score: {hallucinated_score:.4f}")
print(f"  Reason: {hallucinated_reason[:150]}")
print(f"\nThe hallucinated claims about '90-day holiday window' and 'VIP year-long returns'")
print(f"are not in the context, so faithfulness drops significantly.")

-
## 5. HallucinationMetric

### What It Measures

The **HallucinationMetric** checks whether the actual output contains information that *contradicts* or is *not present* in the provided contexts. While similar to Faithfulness, they differ subtly:

| Aspect | FaithfulnessMetric | HallucinationMetric |
|---|---|---|
| Perspective | What proportion of claims are supported? | Does the output contradict the contexts? |
| Score meaning | Higher = more faithful | Higher = less hallucination (confusingly!) |
| Input field | `retrieval_context` | `context` (list of strings) |
| Best for | RAG evaluation | General LLM evaluation |

**Important:** HallucinationMetric uses the `context` parameter (not `retrieval_context`). The score represents how much the output agrees with the context — a score of 1.0 means no hallucination.

In [ ]:
hallucination_metric = HallucinationMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {hallucination_metric.__class__.__name__}")
print(f"Threshold: {hallucination_metric.threshold}")

In [ ]:
# HallucinationMetric uses 'context' instead of 'retrieval_context'
# Let's create test cases with the context parameter

# No hallucination case
no_halluc_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 30-day return policy. Items must be unused with original packaging.",
    context=[
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging."
    ],
)

# Hallucination case — contradicts context
halluc_case = LLMTestCase(
    input="What is the return policy?",
    actual_output="Acme Corp offers a 60-day return policy with no questions asked. All items can be returned even if used, and free return shipping is included for everyone.",
    context=[
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging. Shipping costs for returns are the customer's responsibility."
    ],
)

hallucination_metric.measure(no_halluc_case)
no_halluc_score = hallucination_metric.score
no_halluc_reason = hallucination_metric.reason

hallucination_metric.measure(halluc_case)
halluc_score = hallucination_metric.score
halluc_reason = hallucination_metric.reason

print("HallucinationMetric Comparison")
print("=" * 50)
print(f"No hallucination:   Score = {no_halluc_score:.4f}")
print(f"  Reason: {no_halluc_reason[:150]}")
print(f"\nWith hallucination: Score = {halluc_score:.4f}")
print(f"  Reason: {halluc_reason[:150]}")

### Faithfulness vs. Hallucination — When to Use Which

- **FaithfulnessMetric** is designed specifically for RAG. It checks that all claims in the answer come from `retrieval_context`. Use this as your primary metric.
- **HallucinationMetric** is more general. It checks whether the output contradicts the provided `context`. Use this when you have ground truth context (not necessarily from retrieval).

**Recommendation:** For RAG evaluation, use `FaithfulnessMetric` as your primary hallucination detector. Use `HallucinationMetric` for additional validation or when you have a fixed set of ground truth contexts.

---
## RAGAS Generator Metrics

Now we run the equivalent RAGAS generator metrics on the **same data**.

## 5. Faithfulness

### How Faithfulness Works

Faithfulness measures whether the generated answer is **grounded in the retrieved context**. A faithful answer only contains claims that can be verified from the provided context.

**Algorithm (Claim Extraction + NLI)**:
1. **Claim Extraction**: The LLM extracts individual factual claims from the generated answer
2. **Natural Language Inference (NLI)**: Each claim is checked against the retrieved contexts to determine if it is supported (entailed), contradicted, or neither
3. **Score Calculation**: `Faithfulness = (Number of supported claims) / (Total number of claims)`

A score of 1.0 means every claim in the answer is supported by the context. A score of 0.0 means no claims are supported.

In [ ]:
# Run Faithfulness metric
faithfulness_metric = Faithfulness(llm=evaluator_llm)

faithfulness_results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness_metric],
)

faithfulness_df = faithfulness_results.to_pandas()
metric_col = faithfulness_df.columns[-1]
print(f"Faithfulness Scores (column: '{metric_col}'):")
print("=" * 60)
for idx, row in faithfulness_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row[metric_col]:.3f}")
print(f"\nMean Faithfulness: {faithfulness_df[metric_col].mean():.3f}")

### Interpreting Faithfulness Scores

| Score Range | Interpretation |
|-------------|----------------|
| 0.9 - 1.0  | Highly faithful -- all claims grounded in context |
| 0.7 - 0.9  | Mostly faithful -- minor unsupported claims |
| 0.5 - 0.7  | Partially faithful -- significant unsupported content |
| 0.0 - 0.5  | Low faithfulness -- mostly hallucinated |

Low faithfulness indicates the LLM is adding information not present in the retrieved context (hallucination).

## 6. Answer Relevancy

### How Answer Relevancy Works

Answer Relevancy measures whether the generated answer is **relevant to the user's question**. An answer can be faithful (grounded) but still irrelevant if it does not address what was asked.

**Algorithm (Reverse Question Generation + Embedding Similarity)**:
1. **Question Generation**: Given the answer, the LLM generates N hypothetical questions that the answer could be responding to
2. **Embedding Similarity**: Each generated question is compared to the original question using embedding cosine similarity
3. **Score Calculation**: `Answer Relevancy = mean(cosine_similarity(original_question, generated_question_i))` for all i

A score of 1.0 means the answer perfectly addresses the question. Low scores indicate the answer is off-topic or only partially relevant.

In [ ]:
# Run Answer Relevancy metric
relevancy_metric = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)

relevancy_results = evaluate(
    dataset=ragas_dataset,
    metrics=[relevancy_metric],
)

relevancy_df = relevancy_results.to_pandas()
metric_col = relevancy_df.columns[-1]
print(f"Answer Relevancy Scores (column: '{metric_col}'):")
print("=" * 60)
for idx, row in relevancy_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row[metric_col]:.3f}")
print(f"\nMean Answer Relevancy: {relevancy_df[metric_col].mean():.3f}")

## 10. Answer Correctness

### How Answer Correctness Works

Answer Correctness evaluates the **overall quality** of the generated answer by combining two components:

1. **Factual Correctness (F1 Score)**: Compares claims in the generated answer against the reference answer
   - True Positive (TP): Claims present in both answer and reference
   - False Positive (FP): Claims in answer but not in reference
   - False Negative (FN): Claims in reference but not in answer
   - `F1 = 2 * TP / (2 * TP + FP + FN)`

2. **Semantic Similarity**: Embedding-based similarity between the answer and reference

The final score is a weighted combination:
`Answer Correctness = w1 * Factual_Correctness + w2 * Semantic_Similarity`

By default, the weights are 0.75 for factual and 0.25 for semantic.

In [ ]:
# Run Factual Correctness and Semantic Similarity metrics
factual_correctness_metric = FactualCorrectness(llm=evaluator_llm)
semantic_similarity_metric = SemanticSimilarity(embeddings=evaluator_embeddings)

correctness_results = evaluate(
    dataset=ragas_dataset,
    metrics=[factual_correctness_metric, semantic_similarity_metric],
)

correctness_df = correctness_results.to_pandas()

# Dynamically detect the metric columns (last two columns added by RAGAS)
# The non-metric columns are the sample fields; metric columns are the extras
non_metric_cols = {"user_input", "response", "retrieved_contexts", "reference", "multi_turn_input"}
metric_cols = [c for c in correctness_df.columns if c not in non_metric_cols]
factual_col = metric_cols[-2] if len(metric_cols) >= 2 else metric_cols[0]
semantic_col = metric_cols[-1] if len(metric_cols) >= 2 else metric_cols[-1]

print("Answer Correctness Scores:")
print("=" * 80)
print(f"  {'Query':<45} {factual_col:>16} {semantic_col:>16}")
print("-" * 80)
for idx, row in correctness_df.iterrows():
    query_short = row["user_input"][:43]
    factual = row.get(factual_col, 0)
    semantic = row.get(semantic_col, 0)
    print(f"  [{idx+1:2d}] {query_short:<43} {factual:>8.3f}         {semantic:>8.3f}")
print(f"\nMean {factual_col}: {correctness_df[factual_col].mean():.3f}")
print(f"Mean {semantic_col}:  {correctness_df[semantic_col].mean():.3f}")

---
## Side-by-Side: DeepEval vs RAGAS Generator Metrics

Compare scores from both frameworks on the same test cases.

In [ ]:
# Build comparison table for generator metrics
comparison_data = {
    "Query": [tc.input[:45] + "..." for tc in test_cases],
    "DE Relevancy": ar_scores,
    "DE Faithfulness": faith_scores,
}

# Add RAGAS scores using the dynamically detected column names
try:
    faith_col = faithfulness_df.columns[-1]
    comparison_data["RAGAS Faithfulness"] = faithfulness_df[faith_col].tolist()
except:
    print("RAGAS Faithfulness scores not available")

try:
    rel_col = relevancy_df.columns[-1]
    comparison_data["RAGAS Relevancy"] = relevancy_df[rel_col].tolist()
except:
    print("RAGAS Relevancy scores not available")

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print(f"\nFramework Averages:")
for col in comparison_df.columns[1:]:
    if comparison_df[col].dtype in ['float64', 'int64']:
        print(f"  {col}: {comparison_df[col].mean():.3f}")

-
## 8. Interpreting Generator Metrics

### Answer Relevancy

| Score Range | Interpretation | Action |
|---|---|---|
| 0.9 - 1.0 | Excellent — answer is focused and relevant | None needed |
| 0.7 - 0.9 | Good — mostly relevant, minor tangents | Consider tightening the system prompt |
| 0.5 - 0.7 | Fair — some off-topic content | Revise the prompt template, add instructions to stay focused |
| < 0.5 | Poor — answer misses the point | Fundamental prompt/model issue, investigate specific cases |

### Faithfulness

| Score Range | Interpretation | Action |
|---|---|---|
| 0.9 - 1.0 | Excellent — fully grounded in context | None needed |
| 0.7 - 0.9 | Good — minor unsupported claims | Add "only use provided context" to prompt |
| 0.5 - 0.7 | Concerning — some hallucination | Lower temperature, strengthen grounding instructions |
| < 0.5 | Critical — significant hallucination | Major prompt revision needed, consider a different model |

-
## 9. Summary — Which Metrics to Use When

### Minimum Viable Evaluation (2 metrics)
- **AnswerRelevancyMetric** — is the answer relevant?
- **FaithfulnessMetric** — is the answer grounded?

### Standard RAG Evaluation (4 metrics)
Add retriever metrics:
- **ContextualRelevancyMetric** — are retrieved docs relevant?
- **ContextualRecallMetric** — are we finding all the relevant info?

### Comprehensive Evaluation (5+ metrics)
Add all of the above plus:
- **ContextualPrecisionMetric** — are relevant docs ranked well?
- Consider adding **G-Eval** custom metrics (Notebook 05)

### Decision Tree

```
Is the answer wrong?
  |
  +-- Is it irrelevant to the question?
  |     --> Low AnswerRelevancy. Fix: improve prompt template.
  |
  +-- Is it relevant but factually wrong?
  |     |
  |     +-- Are the retrieved contexts good?
  |     |     --> Low Faithfulness. Fix: LLM is hallucinating. Lower temp, improve prompt.
  |     |
  |     +-- Are the retrieved contexts bad?
  |           --> Low ContextualRecall/Relevancy. Fix: improve retriever.
  |
  +-- Is it correct but incomplete?
        --> Low ContextualRecall. Fix: retrieve more docs.
```

-
## Next Steps

Proceed to **Notebook 05 (G-Eval & Custom Criteria)** for advanced evaluation techniques:
- G-Eval with custom criteria
- Custom deterministic and LLM-based metrics
- Synthetic data generation
- Batch evaluation with hyperparameter tracking
- Safety metrics (Bias, Toxicity)